In [0]:
# Automatically merges small files during the write process
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoOptimize.autoCompact.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "10")

##  Flattens JSON, enforces schema, and appends to the "One Big Table" with an ingestion_time stamp.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("email", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("product_id", IntegerType()),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("price", LongType()),
    StructField("brand", StringType()),
    StructField("quantity", LongType()),
    StructField("total_amount", LongType()),
    StructField("order_date", StringType())
])

In [0]:
bronze_path = "abfss://bronze@mystreamingdb.dfs.core.windows.net/orders_json"
silver_path = "abfss://silver@mystreamingdb.dfs.core.windows.net/orders_data/"
checkpoint_silver_read = "abfss://silver@mystreamingdb.dfs.core.windows.net/schema_location_read/"
checkpoint_silver_write = "abfss://silver@mystreamingdb.dfs.core.windows.net/schema_location_write/"
checkpoint_display = "abfss://silver@mystreamingdb.dfs.core.windows.net/display/"

In [0]:
# AUTO LOADER READ (Bronze)
df_raw = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .option("cloudFiles.schemaLocation", checkpoint_silver_read + "/schema") \
    .option("cloudFiles.inferColumnTypes", "false") \
    .load(bronze_path)


In [0]:
#display(df_raw, checkpointLocation = checkpoint_display)

In [0]:
# 1. Parse the nested "value" string into a structured column
# 2. Select the fields from that nested structure to the top level
df_parsed = df_raw.withColumn("data", from_json(col("value"), schema)).select("data.*")

In [0]:
#display(df_parsed, checkpointLocation = checkpoint_display)

In [0]:
# TRANSFORMATIONS (Silver Cleaning)
df_cleaned = (
    df_parsed \
    .withColumn("order_id", col("order_id").cast("int")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
    .withColumn("product_id", col("product_id").cast("int")) \
    .withColumn("ingestion_time", current_timestamp())
    .filter(col("order_id").isNotNull())
    .filter(col("price").isNotNull())
    .filter(col("quantity").isNotNull())
    .dropDuplicates(["order_id"])
)

In [0]:
#display(df_cleaned, checkpointLocation = checkpoint_display)

In [0]:
# WRITE TO SILVER (Delta Table)
df_cleaned.writeStream \
    .format("delta") \
    .option("checkpointLocation", checkpoint_silver_write) \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .trigger(once=True) \
    .start(silver_path)